In [71]:
import pandas as pd
import numpy as np

In [72]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/2020-jetbrains-python-survey.csv'

In [73]:
jb = pd.read_csv(url, dtype_backend='pyarrow', engine='pyarrow')

In [74]:
jb

,is.python.main,other.lang.None,other.lang.Java,other.lang.JavaScript,other.lang.C/C++,other.lang.PHP,other.lang.C#,other.lang.Ruby,other.lang.Bash / Shell,other.lang.Objective-C,...,job.role.Technical support,job.role.Data analyst,job.role.Business analyst,job.role.Team lead,job.role.Product manager,job.role.CIO / CEO / CTO,job.role.Systems analyst,job.role.Other,age,country.live
0,Yes,None,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,<NA>,Business analyst,<NA>,<NA>,<NA>,<NA>,<NA>,30–39,<NA>
1,Yes,None,Java,JavaScript,<NA>,<NA>,C#,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21–29,India
2,Yes,None,<NA>,<NA>,C/C++,<NA>,<NA>,<NA>,Bash / Shell,<NA>,...,Technical support,Data analyst,<NA>,Team lead,<NA>,<NA>,<NA>,<NA>,30–39,United States
3,Yes,None,<NA>,JavaScript,<NA>,<NA>,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,Yes,None,Java,JavaScript,C/C++,<NA>,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21–29,Italy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54457,Yes,None,<NA>,<NA>,C/C++,<NA>,<NA>,<NA>,Bash / Shell,Objective-C,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Systems analyst,<NA>,21–29,Russian Federation
54458,Yes,None,<NA>,JavaScript,<NA>,<NA>,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
54459,Yes,None,<NA>,JavaScript,<NA>,PHP,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,CIO / CEO / CTO,<NA>,<NA>,21–29,Russian Federation
54460,Yes,None,<NA>,JavaScript,C/C++,PHP,<NA>,<NA>,Bash / Shell,<NA>,...,<NA>,Data analyst,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,30–39,Spain


In [75]:
import collections

In [76]:
counter = collections.defaultdict(list)

In [77]:
for col in sorted(jb.columns):
    period_count = col.count('.')
    if period_count >= 2:
        part_end = 2
    else:
        part_end = 1
    parts = col.split('.')[:part_end]
    counter['.'.join(parts)].append(col)

In [78]:
uniq_cols = []
for cols in counter.values():
    if len(cols) == 1:
        uniq_cols.extend(cols)

In [79]:
uniq_cols

['age',
 'are.you.datascientist',
 'company.size',
 'country.live',
 'employment.status',
 'first.learn.about.main.ide',
 'how.often.use.main.ide',
 'ide.main',
 'is.python.main',
 'job.team',
 'main.purposes',
 'missing.features.main.ide',
 'nps.main.ide',
 'python.years',
 'python2.version.most',
 'python3.version.most',
 'several.projects',
 'team.size',
 'use.python.most',
 'years.of.coding']

In [80]:
(jb
 [uniq_cols]
 .rename(columns=lambda c: c.replace('.', '_'))
 .age
 .value_counts(dropna=False))

age
<NA>           29701
21–29           9710
30–39           7512
40–49           3010
18–20           2567
50–59           1374
60 or older      588
Name: count, dtype: int64[pyarrow]

In [81]:
# poor way of doing it
jb2 = jb[uniq_cols]
age_slice = jb.age.str.slice(0,2)
age_float = age_slice.astype(float)
age_int = age_float.astype('Int64')
jb2['age'] = age_int
jb2['age']

/var/folders/zx/4z_jxwj12xg5lywps2_6jcnm0000gn/T/ipykernel_32460/1171737871.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jb2['age'] = age_int


0          30
1          21
2          30
3        <NA>
4          21
         ... 
54457      21
54458    <NA>
54459      21
54460      30
54461      21
Name: age, Length: 54462, dtype: Int64

In [82]:
(jb[uniq_cols]
.rename(columns=lambda c: c.replace('.','_'))
.assign(age=lambda df_: df_.age
        .str.slice(0,2)
        .replace('', np.nan)
        .astype('int8[pyarrow]')))

,age,are_you_datascientist,company_size,country_live,employment_status,first_learn_about_main_ide,how_often_use_main_ide,ide_main,is_python_main,job_team,main_purposes,missing_features_main_ide,nps_main_ide,python_years,python2_version_most,python3_version_most,several_projects,team_size,use_python_most,years_of_coding
0,30,<NA>,Just me,<NA>,Partially employed by a company / organization,Conference / User Group,Weekly,PyCharm Community Edition,Yes,Work as an external consultant or trainer,For work,"No, it has all the features I need",3,3–5 years,<NA>,Python 3_7,"Yes, I work on many different projects",<NA>,<NA>,1–2 years
1,21,Yes,"More than 5,000",India,Fully employed by a company / organization,School / University,Daily,VS Code,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",8,3–5 years,<NA>,Python 3_6,"Yes, I work on one main and several side projects",2-7 people,Software prototyping,3–5 years
2,30,No,"More than 5,000",United States,Fully employed by a company / organization,Friend / Colleague,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,3–5 years,<NA>,Python 3_6,"Yes, I work on one main and several side projects",<NA>,DevOps / System administration / Writing autom...,3–5 years
3,<NA>,<NA>,<NA>,<NA>,<NA>,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,<NA>,Both for work and personal,Yes – Please list:,10,11+ years,<NA>,Python 3_8,"Yes, I work on many different projects",<NA>,Web development,11+ years
4,21,<NA>,<NA>,Italy,Student,Search engines,Daily,VS Code,Yes,Work on your own project(s) independently,"For personal, educational or side projects","No, it has all the features I need",10,1–2 years,<NA>,Python 3_8,"Yes, I work on one main and several side projects",<NA>,Web development,Less than 1 year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54457,21,No,2–10,Russian Federation,Fully employed by a company / organization,School / University,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,6–10 years,<NA>,Python 3_6,"Yes, I work on many different projects",<NA>,Data analysis,1–2 years
54458,<NA>,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Yes,<NA>,Both for work and personal,<NA>,<NA>,3–5 years,<NA>,Python 3_7,<NA>,<NA>,Web development,1–2 years
54459,21,<NA>,Just me,Russian Federation,Self-employed (a person earning income directl...,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",10,3–5 years,<NA>,Python 3_7,"Yes, I work on many different projects",2-7 people,Web development,6–10 years
54460,30,Yes,51–500,Spain,Fully employed by a company / organization,Search engines,Daily,Other,Yes,Work on your own project(s) independently,Both for work and personal,Yes – Please list:,3,6–10 years,<NA>,Python 3_7,"Yes, I work on many different projects",<NA>,Data analysis,3–5 years


In [83]:
(jb
 [uniq_cols]
 .rename(columns= lambda c: c.replace('.','_'))
 .assign(age=lambda df_: df_.age.str.slice(0,2)
         .astype('int8[pyarrow]'), are_you_datascientist=lambda df_: df_.are_you_datascientist
         .replace({'Yes': '1', 'No': '0', '':'0', 'Other': '0'})
         .astype('bool[pyarrow]'))
         .are_you_datascientist)

0         <NA>
1         True
2        False
3         <NA>
4         <NA>
         ...  
54457    False
54458    False
54459     <NA>
54460     True
54461     <NA>
Name: are_you_datascientist, Length: 54462, dtype: bool[pyarrow]

In [84]:
(jb[uniq_cols]
.rename(columns= lambda c: c.replace('.','_'))
.assign(age=(lambda df_:df_.age.str.slice(0,2)
            .astype('int8[pyarrow]')), are_you_datascientist=lambda df_: df_.are_you_datascientist
            .replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'})
            .astype('bool[pyarrow]'))
            .are_you_datascientist)

0         <NA>
1         True
2        False
3         <NA>
4         <NA>
         ...  
54457    False
54458    False
54459     <NA>
54460     True
54461     <NA>
Name: are_you_datascientist, Length: 54462, dtype: bool[pyarrow]

In [85]:
(jb[uniq_cols]
.rename(columns= lambda c: c.replace('.','_'))
.assign(age=(lambda df_:df_.age.str.slice(0,2)
            .astype('int8[pyarrow]')), are_you_datascientist=lambda df_: df_.are_you_datascientist
            .replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'})
            .astype('bool[pyarrow]')
            )
            .company_size.value_counts())

company_size
51–500             4608
More than 5,000    3635
11–50              3507
2–10               2558
1,001–5,000        1934
Just me            1492
501–1,000          1165
Not sure            526
Name: count, dtype: int64[pyarrow]

In [86]:
(jb[uniq_cols]
.rename(columns= lambda c: c.replace('.','_'))
.assign(age=(lambda df_:df_.age.str.slice(0,2)
            .astype('int8[pyarrow]')), are_you_datascientist=lambda df_: df_.are_you_datascientist
            .replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'})
            .astype('bool[pyarrow]')
            )
            .company_size
            .replace({'Just me': '1', '':pd.NA,'Not sure':pd.NA,'More than 5,000':'5000','2–10':'2','11–50':'11','51–500':'51','501–1,000':'501', '1,001–5,000':'1001'})
            .astype('int64[pyarrow]')
            .value_counts())

company_size
51      4608
5000    3635
11      3507
2       2558
1001    1934
1       1492
501     1165
Name: count, dtype: int64[pyarrow]

In [87]:
(jb[uniq_cols]
 .rename(columns= lambda c: c.replace('.','_'))
 .assign(age=(lambda df_: df_.age.str.slice(0,2)
              .astype('int8[pyarrow]')), are_you_datascientist= lambda df_: df_.are_you_datascientist
              .replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'})
              .astype('bool[pyarrow]'), company_size= lambda df_:df_.company_size.replace({'Just me': '1', '':pd.NA,'Not sure':pd.NA,'More than 5,000':'5000','2–10':'2','11–50':'11','51–500':'51','501–1,000':'501', '1,001–5,000':'1001'})
              .astype('int64[pyarrow]'))
              )

,age,are_you_datascientist,company_size,country_live,employment_status,first_learn_about_main_ide,how_often_use_main_ide,ide_main,is_python_main,job_team,main_purposes,missing_features_main_ide,nps_main_ide,python_years,python2_version_most,python3_version_most,several_projects,team_size,use_python_most,years_of_coding
0,30,<NA>,1,<NA>,Partially employed by a company / organization,Conference / User Group,Weekly,PyCharm Community Edition,Yes,Work as an external consultant or trainer,For work,"No, it has all the features I need",3,3–5 years,<NA>,Python 3_7,"Yes, I work on many different projects",<NA>,<NA>,1–2 years
1,21,True,5000,India,Fully employed by a company / organization,School / University,Daily,VS Code,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",8,3–5 years,<NA>,Python 3_6,"Yes, I work on one main and several side projects",2-7 people,Software prototyping,3–5 years
2,30,False,5000,United States,Fully employed by a company / organization,Friend / Colleague,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,3–5 years,<NA>,Python 3_6,"Yes, I work on one main and several side projects",<NA>,DevOps / System administration / Writing autom...,3–5 years
3,<NA>,<NA>,<NA>,<NA>,<NA>,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,<NA>,Both for work and personal,Yes – Please list:,10,11+ years,<NA>,Python 3_8,"Yes, I work on many different projects",<NA>,Web development,11+ years
4,21,<NA>,<NA>,Italy,Student,Search engines,Daily,VS Code,Yes,Work on your own project(s) independently,"For personal, educational or side projects","No, it has all the features I need",10,1–2 years,<NA>,Python 3_8,"Yes, I work on one main and several side projects",<NA>,Web development,Less than 1 year
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54457,21,False,2,Russian Federation,Fully employed by a company / organization,School / University,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,6–10 years,<NA>,Python 3_6,"Yes, I work on many different projects",<NA>,Data analysis,1–2 years
54458,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Yes,<NA>,Both for work and personal,<NA>,<NA>,3–5 years,<NA>,Python 3_7,<NA>,<NA>,Web development,1–2 years
54459,21,<NA>,1,Russian Federation,Self-employed (a person earning income directl...,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",10,3–5 years,<NA>,Python 3_7,"Yes, I work on many different projects",2-7 people,Web development,6–10 years
54460,30,True,51,Spain,Fully employed by a company / organization,Search engines,Daily,Other,Yes,Work on your own project(s) independently,Both for work and personal,Yes – Please list:,3,6–10 years,<NA>,Python 3_7,"Yes, I work on many different projects",<NA>,Data analysis,3–5 years


In [88]:
jb2 = (jb
        [uniq_cols]
        .rename(columns=lambda c: c.replace('.', '_'))
        .assign(age=lambda df_: df_.age.str.slice(0,2).astype('int8[pyarrow]'),
                are_you_datascientist= lambda df_: df_.are_you_datascientist.replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'}).astype('bool[pyarrow]'),
                company_size = lambda df_: df_.company_size.replace({'Just me': '1', '':pd.NA,'Not sure':pd.NA,'More than 5,000':'5000','2–10':'2','11–50':'11','51–500':'51','501–1,000':'501', '1,001–5,000':'1001'}).astype('int64[pyarrow]'),
                country_live = lambda df_: df_.country_live.astype('category'),
                employment_status= lambda df_: df_.employment_status.fillna('Other').astype('category'),
                is_python_main = lambda df_: df_.is_python_main.astype('category'),
                team_size = lambda df_: df_.team_size.str.split(f'-', n=1, expand=True).iloc[:,0].replace('More than 40 people', '41').where(df_.company_size!=1, '1').replace('', pd.NA).astype('int8[pyarrow]'),
                years_of_coding = lambda df_: df_.years_of_coding.replace('Less than 1 year', '.5').str.extract(r'(?P<years_of_coding>\.?\d+)').astype('float64[pyarrow]'),
                python_years = lambda df_: df_.python_years.replace('Less than 1 year', '.5').str.extract(r'(?P<python_years>\.?\d+)').astype('float64[pyarrow]'), 
                python3_ver = lambda df_: df_.python3_version_most.str.replace('_','.').str.extract(r'(?P<python3_ver>\d\.\d)'),#.astype('float64[pyarrow]'), 
                use_python_most=lambda df_: df_.use_python_most.fillna('Unknown')
                ).drop(columns=['python2_version_most']))

In [89]:
jb2

,age,are_you_datascientist,company_size,country_live,employment_status,first_learn_about_main_ide,how_often_use_main_ide,ide_main,is_python_main,job_team,main_purposes,missing_features_main_ide,nps_main_ide,python_years,python3_version_most,several_projects,team_size,use_python_most,years_of_coding,python3_ver
0,30,<NA>,1,<NA>,Partially employed by a company / organization,Conference / User Group,Weekly,PyCharm Community Edition,Yes,Work as an external consultant or trainer,For work,"No, it has all the features I need",3,3.0,Python 3_7,"Yes, I work on many different projects",1,Unknown,1.0,3.7
1,21,True,5000,India,Fully employed by a company / organization,School / University,Daily,VS Code,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",8,3.0,Python 3_6,"Yes, I work on one main and several side projects",2,Software prototyping,3.0,3.6
2,30,False,5000,United States,Fully employed by a company / organization,Friend / Colleague,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,3.0,Python 3_6,"Yes, I work on one main and several side projects",<NA>,DevOps / System administration / Writing autom...,3.0,3.6
3,<NA>,<NA>,<NA>,<NA>,Other,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,<NA>,Both for work and personal,Yes – Please list:,10,11.0,Python 3_8,"Yes, I work on many different projects",1,Web development,11.0,3.8
4,21,<NA>,<NA>,Italy,Student,Search engines,Daily,VS Code,Yes,Work on your own project(s) independently,"For personal, educational or side projects","No, it has all the features I need",10,1.0,Python 3_8,"Yes, I work on one main and several side projects",1,Web development,0.5,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54457,21,False,2,Russian Federation,Fully employed by a company / organization,School / University,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,6.0,Python 3_6,"Yes, I work on many different projects",<NA>,Data analysis,1.0,3.6
54458,<NA>,False,<NA>,<NA>,Other,<NA>,<NA>,<NA>,Yes,<NA>,Both for work and personal,<NA>,<NA>,3.0,Python 3_7,<NA>,1,Web development,1.0,3.7
54459,21,<NA>,1,Russian Federation,Self-employed (a person earning income directl...,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",10,3.0,Python 3_7,"Yes, I work on many different projects",1,Web development,6.0,3.7
54460,30,True,51,Spain,Fully employed by a company / organization,Search engines,Daily,Other,Yes,Work on your own project(s) independently,Both for work and personal,Yes – Please list:,3,6.0,Python 3_7,"Yes, I work on many different projects",<NA>,Data analysis,3.0,3.7


In [90]:
jb2.columns

Index(['age', 'are_you_datascientist', 'company_size', 'country_live',
       'employment_status', 'first_learn_about_main_ide',
       'how_often_use_main_ide', 'ide_main', 'is_python_main', 'job_team',
       'main_purposes', 'missing_features_main_ide', 'nps_main_ide',
       'python_years', 'python3_version_most', 'several_projects', 'team_size',
       'use_python_most', 'years_of_coding', 'python3_ver'],
      dtype='object')

In [91]:
jb2 = (jb
        [uniq_cols]
        .rename(columns=lambda c: c.replace('.', '_'))
        .assign(age=lambda df_: df_.age.str.slice(0,2).astype('int8[pyarrow]'),
                are_you_datascientist= lambda df_: df_.are_you_datascientist.replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'}).astype('bool[pyarrow]'),
                company_size = lambda df_: df_.company_size.replace({'Just me': '1', '':pd.NA,'Not sure':pd.NA,'More than 5,000':'5000','2–10':'2','11–50':'11','51–500':'51','501–1,000':'501', '1,001–5,000':'1001'}).astype('int64[pyarrow]'),
                country_live = lambda df_: df_.country_live.astype('category'),
                employment_status= lambda df_: df_.employment_status.fillna('Other').astype('category'),
                is_python_main = lambda df_: df_.is_python_main.astype('category'),
                team_size = lambda df_: df_.team_size.str.split(f'-', n=1, expand=True).iloc[:,0].replace('More than 40 people', '41').where(df_.company_size!=1, '1').replace('', pd.NA).astype('int8[pyarrow]'),
                years_of_coding = lambda df_: df_.years_of_coding.replace('Less than 1 year', '.5').str.extract(r'(?P<years_of_coding>\.?\d+)').astype('float64[pyarrow]'),
                python_years = lambda df_: df_.python_years.replace('Less than 1 year', '.5').str.extract(r'(?P<python_years>\.?\d+)').astype('float64[pyarrow]'), 
                python3_ver = lambda df_: df_.python3_version_most.str.replace('_','.').str.extract(r'(?P<python3_ver>\d\.\d)'),#.astype('float64[pyarrow]'), 
                use_python_most=lambda df_: df_.use_python_most.fillna('Unknown')
                )#.drop(columns=['python2_version_most'])
                .loc[:, ['age', 'are_you_datascientist', 'company_size', 'country_live',
       'employment_status', 'first_learn_about_main_ide',
       'how_often_use_main_ide', 'ide_main', 'is_python_main', 'job_team',
       'main_purposes', 'missing_features_main_ide', 'nps_main_ide',
       'python_years', 'python3_version_most', 'several_projects', 'team_size',
       'use_python_most', 'years_of_coding', 'python3_ver']]
                )

In [92]:
jb2

,age,are_you_datascientist,company_size,country_live,employment_status,first_learn_about_main_ide,how_often_use_main_ide,ide_main,is_python_main,job_team,main_purposes,missing_features_main_ide,nps_main_ide,python_years,python3_version_most,several_projects,team_size,use_python_most,years_of_coding,python3_ver
0,30,<NA>,1,<NA>,Partially employed by a company / organization,Conference / User Group,Weekly,PyCharm Community Edition,Yes,Work as an external consultant or trainer,For work,"No, it has all the features I need",3,3.0,Python 3_7,"Yes, I work on many different projects",1,Unknown,1.0,3.7
1,21,True,5000,India,Fully employed by a company / organization,School / University,Daily,VS Code,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",8,3.0,Python 3_6,"Yes, I work on one main and several side projects",2,Software prototyping,3.0,3.6
2,30,False,5000,United States,Fully employed by a company / organization,Friend / Colleague,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,3.0,Python 3_6,"Yes, I work on one main and several side projects",<NA>,DevOps / System administration / Writing autom...,3.0,3.6
3,<NA>,<NA>,<NA>,<NA>,Other,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,<NA>,Both for work and personal,Yes – Please list:,10,11.0,Python 3_8,"Yes, I work on many different projects",1,Web development,11.0,3.8
4,21,<NA>,<NA>,Italy,Student,Search engines,Daily,VS Code,Yes,Work on your own project(s) independently,"For personal, educational or side projects","No, it has all the features I need",10,1.0,Python 3_8,"Yes, I work on one main and several side projects",1,Web development,0.5,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54457,21,False,2,Russian Federation,Fully employed by a company / organization,School / University,Daily,Vim,Yes,Work on your own project(s) independently,Both for work and personal,"No, it has all the features I need",10,6.0,Python 3_6,"Yes, I work on many different projects",<NA>,Data analysis,1.0,3.6
54458,<NA>,False,<NA>,<NA>,Other,<NA>,<NA>,<NA>,Yes,<NA>,Both for work and personal,<NA>,<NA>,3.0,Python 3_7,<NA>,1,Web development,1.0,3.7
54459,21,<NA>,1,Russian Federation,Self-employed (a person earning income directl...,Friend / Colleague,Daily,PyCharm Professional Edition,Yes,Work in a team,Both for work and personal,"No, it has all the features I need",10,3.0,Python 3_7,"Yes, I work on many different projects",1,Web development,6.0,3.7
54460,30,True,51,Spain,Fully employed by a company / organization,Search engines,Daily,Other,Yes,Work on your own project(s) independently,Both for work and personal,Yes – Please list:,3,6.0,Python 3_7,"Yes, I work on many different projects",<NA>,Data analysis,3.0,3.7


In [93]:
jb2.team_size.value_counts(dropna=False) # 6887 missing values

team_size
1       37055
2        7740
<NA>     6887
8        1798
13        587
21        221
41        174
Name: count, dtype: int64[pyarrow]

In [94]:
def limit_index_length(df, max_length):
    return df.rename(index= lambda x: x[:max_length])

In [95]:
(jb2
 .query('team_size.isna()')
 .employment_status
 .value_counts(dropna=False)
 #.pipe(limit_index_length, max_length=40)
 .index
 )

CategoricalIndex([                                                    'Fully employed by a company / organization',
                                                                                                 'Working student',
                                                                  'Partially employed by a company / organization',
                  'Self-employed (a person earning income directly from one's own business, trade, or profession)',
                  'Freelancer (a person pursuing a profession without a long-term commitment to any one employer)',
                                                                                                           'Other',
                                                                                                         'Retired',
                                                                                                         'Student'],
                 categories=[Freelancer (a person pursuing a profession

In [96]:
(jb2
 .query('team_size.isna()')
 .employment_status
 .value_counts(dropna=False)
 .pipe(limit_index_length, max_length=40)
 .index
 )

Index(['Fully employed by a company / organizati', 'Working student',
       'Partially employed by a company / organi',
       'Self-employed (a person earning income d',
       'Freelancer (a person pursuing a professi', 'Other', 'Retired',
       'Student'],
      dtype='object', name='employment_status')

In [97]:
import catboost as cb
cb.__version__

'1.2.10'


While CatBoost works with data from pandas dataframes, however, it is picky about the

input that it supports. It doesn't like native pandas types (like 'Int64' of 'category'), so

I'm going to make a function, prep_for_ml, that uses two dictionary comprehensions to

change the column types when we make our predictions.

In [98]:
def prep_for_ml(df):
    # remove pandas types
    return (df
            .assign(**df.select_dtypes(['number','bool'])
                    .astype('float[pyarrow]').astype(float),
                    **{col:df[col].astype(str).fillna('')
                       for col in df.select_dtypes(['object','category','string'])})
                                                    )

In [99]:
def predict_col(df, col):
    df = prep_for_ml(df)
    missing = df.query(f'~{col}.isna()')
    cat_idx = [i for i, typ in enumerate(df.drop(columns=[col]).dtypes)
               if str(typ) == 'object']
    X = (missing
         .drop(columns=[col])
         .values)
    
    y = missing[col]
    model = cb.CatBoostRegressor(iterations=20, cat_features=cat_idx)
    model.fit(X, y, cat_features = cat_idx)
    pred = model.predict(df.drop(columns=[col]))
    return df[col].where(~df[col].isna(), pred)

In [100]:
jb2.team_size.value_counts(dropna=False) # 6887 missing values

team_size
1       37055
2        7740
<NA>     6887
8        1798
13        587
21        221
41        174
Name: count, dtype: int64[pyarrow]

In [101]:
jb2 = (jb
        [uniq_cols]
        .rename(columns=lambda c: c.replace('.', '_'))
        .assign(age=lambda df_: df_.age.str.slice(0,2).astype('int8[pyarrow]'),
                are_you_datascientist= lambda df_: df_.are_you_datascientist.replace({'Yes': '1', 'No': '0', '': '0', 'Other':'0'}).astype('bool[pyarrow]'),
                company_size = lambda df_: df_.company_size.replace({'Just me': '1', '':pd.NA,'Not sure':pd.NA,'More than 5,000':'5000','2–10':'2','11–50':'11','51–500':'51','501–1,000':'501', '1,001–5,000':'1001'}).astype('int64[pyarrow]'),
                country_live = lambda df_: df_.country_live.astype('category'),
                employment_status= lambda df_: df_.employment_status.fillna('Other').astype('category'),
                is_python_main = lambda df_: df_.is_python_main.astype('category'),
                team_size = lambda df_: df_.team_size.str.split(f'-', n=1, expand=True).iloc[:,0].replace('More than 40 people', '41').where(df_.company_size!=1, '1').replace('', pd.NA).astype('int8[pyarrow]'),
                years_of_coding = lambda df_: df_.years_of_coding.replace('Less than 1 year', '.5').str.extract(r'(?P<years_of_coding>\.?\d+)').astype('float64[pyarrow]'),
                python_years = lambda df_: df_.python_years.replace('Less than 1 year', '.5').str.extract(r'(?P<python_years>\.?\d+)').astype('float64[pyarrow]'), 
                python3_ver = lambda df_: df_.python3_version_most.str.replace('_','.').str.extract(r'(?P<python3_ver>\d\.\d)'),#.astype('float64[pyarrow]'), 
                use_python_most=lambda df_: df_.use_python_most.fillna('Unknown')
                )
        .assign(
            team_size= lambda df_: predict_col(df_, 'team_size')
            .astype(int)
        )
                .loc[:, ['age', 'are_you_datascientist', 'company_size', 'country_live',
       'employment_status', 'first_learn_about_main_ide',
       'how_often_use_main_ide', 'ide_main', 'is_python_main', 'job_team',
       'main_purposes', 'missing_features_main_ide', 'nps_main_ide',
       'python_years', 'python3_version_most', 'several_projects', 'team_size',
       'use_python_most', 'years_of_coding', 'python3_ver']]
                )

Learning rate set to 0.5
0:	learn: 2.9758568	total: 8.04ms	remaining: 153ms
1:	learn: 2.8841040	total: 17.6ms	remaining: 158ms
2:	learn: 2.8443484	total: 24.8ms	remaining: 141ms
3:	learn: 2.8105584	total: 31.4ms	remaining: 126ms
4:	learn: 2.7922983	total: 37.1ms	remaining: 111ms
5:	learn: 2.7803329	total: 42.9ms	remaining: 100ms
6:	learn: 2.7756137	total: 49.2ms	remaining: 91.3ms
7:	learn: 2.7706510	total: 55ms	remaining: 82.5ms
8:	learn: 2.7571563	total: 62ms	remaining: 75.8ms
9:	learn: 2.7564631	total: 68.6ms	remaining: 68.6ms
10:	learn: 2.7503591	total: 74.4ms	remaining: 60.8ms
11:	learn: 2.7494745	total: 80.1ms	remaining: 53.4ms
12:	learn: 2.7481258	total: 86.2ms	remaining: 46.4ms
13:	learn: 2.7477180	total: 92.3ms	remaining: 39.6ms
14:	learn: 2.7449738	total: 99.1ms	remaining: 33ms
15:	learn: 2.7409940	total: 105ms	remaining: 26.1ms
16:	learn: 2.7408640	total: 110ms	remaining: 19.4ms
17:	learn: 2.7365108	total: 116ms	remaining: 12.9ms
18:	learn: 2.7346780	total: 122ms	remaining: 6

In [102]:
jb2.team_size.value_counts(dropna=False)

team_size
1     37949
2      8192
8      1868
3      1580
4      1546
5      1289
6       673
13      600
7       258
21      221
41      174
9        59
10       23
11       12
12        7
15        6
0         2
17        2
16        1
Name: count, dtype: int64